In [ ]:
!nvidia-smi


In [ ]:
# ============================================================
# Mandatory GPU check — before continuing, make sure we're really
# connected to a GPU. The 8-billion-parameter diffusion model is
# practically unusable on CPU, so if there's no GPU right now, it's
# better to stop here rather than waste the next 15-20 minutes
# (installation, dataset upload, graph building) only to find out at
# the very last step that no GPU was connected.
# ============================================================
import subprocess

gpu_check = subprocess.run(["nvidia-smi"], capture_output=True, text=True)

if gpu_check.returncode != 0:
    raise RuntimeError(
        "\n\n"
        "🛑 No GPU connected! Before continuing, do the following:\n"
        "   1) From the menu: Runtime > Change runtime type\n"
        "   2) Under Hardware accelerator: select T4 GPU (not None)\n"
        "   3) Click Save and confirm the runtime reconnects\n"
        "   4) Run this cell again\n\n"
        "If you still see this error after selecting GPU, your Colab account's "
        "daily free GPU quota has probably run out — wait a few hours or use a "
        "different account/Colab Pro.\n"
    )
else:
    print("✅ GPU is connected — you can safely continue.")
    print(gpu_check.stdout.split(chr(10))[8] if len(gpu_check.stdout.split(chr(10))) > 8 else "")


In [ ]:
# ============================================================
# Overall project configuration — these values are used in several
# of the following cells
# ============================================================

NEO4J_PASSWORD_COLAB = "Colab.Password123"   # optional, feel free to change

# If you don't have a GPU, or don't want the diffusion model to load at
# all, set this to False.
DIFFUSION_ENABLED_COLAB = True

# STRICT MODE: if True, any failure to load/run the diffusion model
# raises an explicit error instead of silently degrading to the
# template-based response (useful for debugging — you'll know exactly
# where the problem is). For a final/demo run where the whole CLI
# shouldn't stop, set this to False.
STRICT_DIFFUSION_MODE_COLAB = True

print("✅ Project configuration initialized.")
print(f"   NEO4J_PASSWORD_COLAB         = {NEO4J_PASSWORD_COLAB}")
print(f"   DIFFUSION_ENABLED_COLAB      = {DIFFUSION_ENABLED_COLAB}")
print(f"   STRICT_DIFFUSION_MODE_COLAB  = {STRICT_DIFFUSION_MODE_COLAB}")


In [ ]:
%%bash
# Bug fix: if this cell is run again (e.g. after an error in a later
# cell), the previous mv/tar would fail with "File exists" because
# /opt/neo4j already existed. Now we check beforehand whether it's
# already installed.
if [ -d /opt/neo4j ]; then
    echo "✅ Neo4j is already installed — skipping reinstall."
else
    apt-get update -qq
    apt-get install -y openjdk-17-jdk -qq
    wget -q https://dist.neo4j.org/neo4j-community-5.22.0-unix.tar.gz
    tar -xzf neo4j-community-5.22.0-unix.tar.gz
    mv neo4j-community-5.22.0 /opt/neo4j
    echo "✅ Neo4j installation complete"
fi


In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

result = os.system(
    f'/opt/neo4j/bin/neo4j-admin dbms set-initial-password "{NEO4J_PASSWORD_COLAB}"'
)
if result == 0:
    print(f"✅ Neo4j password set successfully: {NEO4J_PASSWORD_COLAB}")
else:
    print("❌ Setting the password failed — check the log above.")


In [ ]:
import subprocess, time, urllib.request, json

neo4j_proc = subprocess.Popen(
    ["/opt/neo4j/bin/neo4j", "console"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

print("⏳ Waiting for Neo4j to come up...")

up = False
for attempt in range(15):
    time.sleep(3)
    try:
        with urllib.request.urlopen("http://localhost:7474", timeout=3) as resp:
            info = json.loads(resp.read().decode())
            up = True
            break
    except Exception:
        continue

if up:
    print("✅ Neo4j came up successfully.")
    print(f"   Version: {info.get('neo4j_version')} | Edition: {info.get('neo4j_edition')}")
    print(f"   Bolt endpoint: {info.get('bolt_direct')}")
else:
    print("❌ Neo4j did not come up after 45 seconds. Run the Section 1 cells "
          "from the beginning (also try Runtime > Restart session).")


In [ ]:
# Note: the pip warning about sentence-transformers (if you see it) is
# harmless — this library is never actually used/imported in our
# project; it just happens to already be installed on the base Colab
# image, and pip's dependency resolver warns about its version, but
# transformers==4.38.2 is never actually overridden (confirmed in the
# report at the bottom of this same cell).
#
# Bug fix (new): the previous bitsandbytes==0.42.0 pin crashes on the
# current Colab image with "CUDA Setup failed despite GPU being
# available". The cause isn't that the GPU or CUDA is actually broken;
# version 0.42 uses an old, fragile module called cuda_setup that
# searches hard-coded paths for libcudart.so libraries and can't find
# them on the newer CUDA layout in recent Colab images (which has
# changed since early 2024 when this version was pinned), raising a
# RuntimeError. This is a known, closed bug in the bitsandbytes project
# itself, not an issue in this notebook. From version 0.43 onward, the
# whole CUDA detection was rewritten (it no longer has that fragile
# module) and uses torch's own CUDA info directly, so bitsandbytes was
# updated here to >=0.43.1. accelerate was also updated to a matching
# version (0.30.1) so incompatible APIs don't collide again.
# transformers stays pinned exactly at 4.38.2 (required by the official
# LLaDA repo). If this exact combination still returns the old
# ".to is not supported for 4-bit models" error, the two-attempt retry
# logic already present in DiffusionResponder._lazy_load
# (low_cpu_mem_usage=True then False) automatically works around it.
!pip install -q neo4j rapidfuzz ddgs "transformers==4.38.2" "accelerate==0.30.1" "bitsandbytes>=0.43.1" sentencepiece einops psutil spacy
!python -m spacy download en_core_web_sm -q

import importlib
packages = ["neo4j", "rapidfuzz", "ddgs", "transformers", "accelerate", "bitsandbytes", "torch", "spacy"]
print("Library installation report:")
all_ok = True
for pkg in packages:
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, "__version__", "unknown")
        print(f"  ✅ {pkg:15s} version: {ver}")
    # Bug fix: on an incompatible environment, bitsandbytes throws a
    # RuntimeError (from inside cextension.py) instead of an
    # ImportError, and it used to stop the whole cell/kernel. Catching
    # the general Exception instead of only ImportError lets this
    # report continue for every library regardless of the error type,
    # and prints the real issue.
    except Exception as e:
        print(f"  ❌ {pkg:15s} failed to import! ({type(e).__name__}: {e})")
        all_ok = False

import spacy
try:
    _nlp_check = spacy.load("en_core_web_sm")
    print("  ✅ spaCy model en_core_web_sm loaded successfully.")
except OSError:
    print("  ❌ spaCy model en_core_web_sm did not download — run the cell again.")
    all_ok = False

try:
    import transformers, accelerate, bitsandbytes
    version_checks = [
        ("transformers", transformers.__version__, "4.38.2"),
        ("accelerate", accelerate.__version__, "0.30.1"),
        ("bitsandbytes", bitsandbytes.__version__, "0.43.1"),
    ]
    mismatch = False
    for name, actual, expected in version_checks:
        # Bug fix: bitsandbytes is pinned with ">=", so a version
        # *newer* than 0.43.1 may be installed, which is completely
        # normal and harmless; only an *older* version (which has the
        # CUDA Setup bug) should trigger a warning.
        if name == "bitsandbytes":
            actual_tuple = tuple(int(p) for p in actual.split(".")[:2])
            if actual_tuple < (0, 43):
                print(f"\n⚠️ The installed bitsandbytes version ({actual}) is older than "
                      f"0.43 — the same CUDA Setup bug may occur again.")
                mismatch = True
            continue
        if actual != expected:
            print(f"\n⚠️ The installed {name} version ({actual}) is not exactly {expected} — "
                  f"loading the diffusion model (Section 8) may fail.")
            mismatch = True
    if mismatch:
        print("Fix: run Runtime > Restart session and run this cell again.")
    else:
        print(f"\n✅ transformers/accelerate/bitsandbytes are installed on versions "
              f"compatible with and matching LLaDA's era.")
except Exception as e:
    # Bug fix: previously, if this same import (a second time, for
    # version_checks) crashed with a RuntimeError, none of this cell's
    # later prints (GPU status and final summary) would run. Now the
    # error is caught and reported, and the cell keeps running.
    print(f"\n❌ Checking the transformers/accelerate/bitsandbytes versions failed: "
          f"{type(e).__name__}: {e}")
    all_ok = False

import torch
print(f"\n  GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU name: {torch.cuda.get_device_name(0)}")

print("\n✅ All libraries installed." if all_ok else "\n⚠️ Some libraries were not installed, see the log above.")


In [ ]:
import os, shutil

os.makedirs("datasets", exist_ok=True)

required_files = ["tmdb_5000_movies.csv", "tmdb_5000_credits.csv", "imdb_top_1000.csv"]

# Persistent cache on Google Drive: once uploaded here, the datasets survive
# across sessions and reruns, so you don't have to re-upload them every time.
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DATASET_DIR = "/content/drive/MyDrive/GraphRAG_Movie_Datasets"
os.makedirs(DRIVE_DATASET_DIR, exist_ok=True)

cached = [f for f in required_files if os.path.exists(f"{DRIVE_DATASET_DIR}/{f}")]

if len(cached) == len(required_files):
    print(f"✅ Found all datasets cached in Google Drive ({DRIVE_DATASET_DIR}) — no upload needed.")
    for f in required_files:
        shutil.copy(f"{DRIVE_DATASET_DIR}/{f}", f"datasets/{f}")
else:
    from google.colab import files
    print("Datasets not found in Google Drive cache.")
    print("Please select the files tmdb_5000_movies.csv, tmdb_5000_credits.csv and imdb_top_1000.csv:")
    uploaded = files.upload()

    for fname in uploaded:
        os.rename(fname, f"datasets/{fname}")

    # Save a copy to Drive so future runs (even in a brand-new session) can
    # skip the upload step entirely.
    for f in required_files:
        path = f"datasets/{f}"
        if os.path.exists(path):
            shutil.copy(path, f"{DRIVE_DATASET_DIR}/{f}")
    print(f"\n✅ Datasets cached to Google Drive ({DRIVE_DATASET_DIR}) for future runs.")

print("\nDatasets report:")
missing = []
for f in required_files:
    path = f"datasets/{f}"
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"  ✅ {f}  ({size_mb:.2f} MB)")
    else:
        print(f"  ❌ {f}  — not found!")
        missing.append(f)

if missing:
    print(f"\n⚠️ The following files are still missing, run the cell again and select all of them: {missing}")
else:
    print("\n✅ All datasets are ready.")


In [ ]:
import pandas as pd
import ast

movies = pd.read_csv('datasets/tmdb_5000_movies.csv')
credits = pd.read_csv('datasets/tmdb_5000_credits.csv')

tmdb = movies.merge(credits, on="title")
tmdb = tmdb[['title', 'genres', 'cast', 'crew', 'release_date']]

print(f"✅ Initial merge complete — number of rows: {len(tmdb)}")
tmdb.head(3)


In [ ]:
def get_director(crew):
    try:
        crew = ast.literal_eval(crew)
        for person in crew:
            if person['job'] == 'Director':
                return person['name']
    except Exception:
        pass
    return None

def get_cast(cast):
    try:
        cast = ast.literal_eval(cast)
        return [actor['name'] for actor in cast[:5]]
    except Exception:
        return []

def extract_names(text):
    try:
        text = ast.literal_eval(text)
        return [item['name'] for item in text]
    except Exception:
        return []

tmdb['director'] = tmdb['crew'].apply(get_director)
tmdb['actors'] = tmdb['cast'].apply(get_cast)
tmdb['genres'] = tmdb['genres'].apply(extract_names)
tmdb['year'] = pd.to_datetime(tmdb['release_date'], errors='coerce').dt.year.astype('Int64')
tmdb = tmdb.drop(columns=['cast', 'crew', 'release_date'])

print("✅ Extraction of director/actors/genres/year complete.")
print(f"   Number of movies with no director found: {tmdb['director'].isna().sum()}")
print(f"   Number of movies with no year found: {tmdb['year'].isna().sum()}")
tmdb.head(3)


In [ ]:
imdb = pd.read_csv('datasets/imdb_top_1000.csv')

# KeyError fix: the common, widely-used version of this dataset on
# Kaggle (harshitshankhdhar) uses the column names Series_Title /
# Released_Year / IMDB_Rating / Star1..Star4, not
# title/year/imdb_rating/actor1..actor4 that the earlier code assumed
# directly and always crashed with a KeyError. Here the mapping is
# only applied to columns that actually exist under the old names, so
# that if the user uploaded a different version of the file with the
# column names already correct, it's left untouched.
column_rename_map = {
    "Series_Title": "title",
    "Released_Year": "year",
    "IMDB_Rating": "imdb_rating",
    "Star1": "actor1",
    "Star2": "actor2",
    "Star3": "actor3",
    "Star4": "actor4",
}
imdb = imdb.rename(columns={k: v for k, v in column_rename_map.items() if k in imdb.columns})

required_imdb_cols = ["title", "year", "imdb_rating", "actor1", "actor2", "actor3", "actor4"]
missing_cols = [c for c in required_imdb_cols if c not in imdb.columns]
if missing_cols:
    raise KeyError(
        f"The following columns were not found in imdb_top_1000.csv: {missing_cols}. "
        f"Columns present in your file: {list(imdb.columns)}"
    )

imdb['actors_imdb'] = imdb[['actor1', 'actor2', 'actor3', 'actor4']].values.tolist()
# Bug fix: in this same well-known dataset, a few rows have a
# non-numeric value (e.g. a Certificate) in the year column due to the
# original scraping error; errors='coerce' turns them into NaN instead
# of stopping the whole cell with a ValueError.
imdb['year'] = pd.to_numeric(imdb['year'], errors='coerce').astype('Int64')
imdb = imdb[['title', 'year', 'imdb_rating']]

final_data = tmdb.merge(imdb, on=['title', 'year'], how='left')
final_data = final_data[['title', 'genres', 'actors', 'director', 'year', 'imdb_rating']]

print(f"✅ Final merge with IMDB complete.")
print(f"   Final dataset shape: {final_data.shape}")
print(f"   Number of movies with an IMDB rating found: {final_data['imdb_rating'].notna().sum()} out of {len(final_data)}")
final_data.head(5)


In [ ]:
from neo4j import GraphDatabase
import time

NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"

class MovieGraph:
    def __init__(self, uri, user, password):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))

    def close(self):
        self.driver.close()

    def add_movies(self, df, report_every=500):
        with self.driver.session() as session:
            start = time.time()
            for i, (_, row) in enumerate(df.iterrows()):
                genres = row['genres'] if isinstance(row['genres'], list) and row['genres'] else ['none']
                actors = row['actors'] if isinstance(row['actors'], list) and row['actors'] else ['none']

                director = row['director']
                if pd.isna(director) or director is None:
                    director = "none"

                title = row['title']
                if pd.isna(title) or title is None:
                    title = "Unknown Movie"

                year = row['year']
                year = 0 if pd.isna(year) else int(year)

                imdb_rating = row['imdb_rating']
                imdb_rating = 0.0 if pd.isna(imdb_rating) else float(imdb_rating)

                session.execute_write(
                    self._create_full_movie, title, year, imdb_rating, genres, actors, director
                )

                if (i + 1) % report_every == 0:
                    elapsed = time.time() - start
                    print(f"  ... {i + 1}/{len(df)} movies processed ({elapsed:.0f} seconds elapsed)")

    @staticmethod
    def _create_full_movie(tx, title, year, rating, genres, actors, director):
        tx.run("MERGE (m:Movie {title: $title})", title=title)

        tx.run("""
            MERGE (y:Year {year: $year})
            WITH y
            MATCH (m:Movie {title: $title})
            MERGE (m)-[:IN_YEAR]->(y)
        """, year=year, title=title)

        rating_int = int(rating)
        tx.run("""
            MERGE (rating:RATING {rating: $rating_int})
            WITH rating
            MATCH (m:Movie {title: $title})
            MERGE (m)-[r:WITH_POINTS]->(rating)
            SET r.real_rating = $rating
        """, title=title, rating=rating, rating_int=rating_int)

        for g in genres:
            tx.run("""
                MERGE (genre:Genre {name: $g})
                WITH genre
                MATCH (m:Movie {title: $title})
                MERGE (m)-[:HAS_GENRE]->(genre)
            """, g=g, title=title)

        for a in actors:
            tx.run("""
                MERGE (actor:Actor {name: $a})
                WITH actor
                MATCH (m:Movie {title: $title})
                MERGE (actor)-[:ActED_IN]->(m)
            """, a=a, title=title)

        tx.run("""
            MERGE (d:Director {name: $director})
            WITH d
            MATCH (m:Movie {title: $title})
            MERGE (d)-[:DIRECTED]->(m)
        """, director=director, title=title)


print(f"🔨 Starting graph construction for {len(final_data)} movies — this will take a few minutes...")
app = MovieGraph(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD_COLAB)
app.add_movies(final_data)
app.close()
print("✅ Graph construction complete!")


In [ ]:
# Final report: number of nodes and relationships built in the graph
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD_COLAB))
with driver.session() as session:
    counts = {}
    for label in ["Movie", "Actor", "Director", "Genre", "Year", "RATING"]:
        counts[label] = session.run(f"MATCH (n:{label}) RETURN count(n) AS c").single()["c"]
    rel_count = session.run("MATCH ()-[r]->() RETURN count(r) AS c").single()["c"]
driver.close()

print("📊 Final knowledge graph report:")
for label, c in counts.items():
    print(f"   {label:10s}: {c:>6,} nodes")
print(f"   {'relationships':10s}: {rel_count:>6,} relationships")

if counts["Movie"] == 0:
    print("\n❌ There are no movies in the graph! Check the Section 4 and 5 cells.")
else:
    print("\n✅ The knowledge graph was built and populated successfully.")


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD_COLAB))
with driver.session() as session:
    result = session.run("""
        MATCH (m:Movie)
        OPTIONAL MATCH (m)-[:HAS_GENRE]->(g:Genre)
        OPTIONAL MATCH (a:Actor)-[:ActED_IN]->(m)
        OPTIONAL MATCH (d:Director)-[:DIRECTED]->(m)
        WITH m, collect(DISTINCT g.name)[0..2] AS genres,
                collect(DISTINCT a.name)[0..3] AS actors,
                d.name AS director
        WHERE size(genres) > 0 AND size(actors) > 0
        RETURN m.title AS title, genres, actors, director
        LIMIT 10
    """)
    sample = [record.data() for record in result]
driver.close()

print(f"✅ Found {len(sample)} sample movies to display the graph.")

G = nx.Graph()
node_colors = []
node_labels = {}

for row in sample:
    movie = row["title"]
    G.add_node(movie, kind="movie")
    for g in row["genres"]:
        if g:
            G.add_node(g, kind="genre")
            G.add_edge(movie, g)
    for a in row["actors"]:
        if a:
            G.add_node(a, kind="actor")
            G.add_edge(movie, a)
    if row["director"] and row["director"] != "none":
        G.add_node(row["director"], kind="director")
        G.add_edge(movie, row["director"])

color_map = {"movie": "#e74c3c", "genre": "#3498db", "actor": "#2ecc71", "director": "#f39c12"}
colors = [color_map[G.nodes[n]["kind"]] for n in G.nodes]
sizes = [1200 if G.nodes[n]["kind"] == "movie" else 500 for n in G.nodes]

plt.figure(figsize=(16, 11))
pos = nx.spring_layout(G, k=0.6, seed=42)
nx.draw_networkx_nodes(G, pos, node_color=colors, node_size=sizes, alpha=0.9)
nx.draw_networkx_edges(G, pos, alpha=0.3)
nx.draw_networkx_labels(G, pos, font_size=8, font_family="DejaVu Sans")

legend_elements = [
    plt.Line2D([0], [0], marker='o', color='w', label=lbl, markerfacecolor=c, markersize=12)
    for lbl, c in [("Movie", "#e74c3c"), ("Genre", "#3498db"), ("Actor", "#2ecc71"), ("Director", "#f39c12")]
]
plt.legend(handles=legend_elements, loc="upper right", fontsize=11)
plt.title(f"Sample of the built knowledge graph ({len(sample)} movies)", fontsize=14)
plt.axis("off")
plt.tight_layout()
plt.savefig("graph_sample.png", dpi=120, bbox_inches="tight")
plt.show()

print("✅ Graph image displayed and saved to graph_sample.png.")


In [ ]:
%%writefile graphrag_movie_cli.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
GraphRAG Movie Assistant — Interactive Terminal CLI
=====================================================

Architecture:
  1) Orchestrator      -> extracts entities (genre / actor / director / year) from the
                           free-text prompt using Regex + dictionary/fuzzy matching
                           (NOT the diffusion model — diffusion LLMs are bad at emitting
                           structured Cypher, so structured parsing is done in code).
  2) Neo4j layer        -> builds a Cypher query from the extracted entities and runs it
                           against the local knowledge graph.
  3) Routing layer       -> if the local graph returns nothing, falls back to a web
                           search (DuckDuckGo by default, Tavily if you set an API key).
  4) Diffusion LLM layer -> the retrieved context (local or web) is stuffed into a
                           prompt template and handed to a diffusion language model
                           (e.g. LLaDA-8B) loaded via `transformers` to produce the
                           final natural-language answer.
  5) CLI layer           -> a REPL loop ("User: ") that keeps running until the user
                           types exit/quit, then closes the Neo4j driver safely.

Graph schema this script assumes:
  (:Movie {title})
  (:Movie)-[:IN_YEAR]->(:Year {year})
  (:Movie)-[:WITH_POINTS {real_rating}]->(:RATING {rating})
  (:Movie)-[:HAS_GENRE]->(:Genre {name})
  (:Actor)-[:ActED_IN]->(:Movie)
  (:Director)-[:DIRECTED]->(:Movie)
"""

import re
import sys
import traceback
from typing import Dict, List, Optional, Any

from neo4j import GraphDatabase

# --------------------------------------------------------------------------------------
# 0. CONFIGURATION — these placeholders are filled in automatically by the notebook cell
#    that writes this file.
# --------------------------------------------------------------------------------------

NEO4J_URI = "bolt://localhost:7687"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = "__NEO4J_PASSWORD__"

# Whether to attempt loading the diffusion LLM at all. Set to False (via the
# notebook config cell) to always use the fast template fallback without loading
# any model.
DIFFUSION_ENABLED = __DIFFUSION_ENABLED__

# STRICT mode: if True, a failure to load/run the diffusion model raises an
# explicit exception instead of silently degrading to the template fallback. Useful
# while actively debugging the diffusion layer, since it forces the real error to
# surface immediately instead of being swallowed. Set to False for a live demo /
# final submission run, so a transient failure doesn't kill the whole CLI session.
STRICT_DIFFUSION_MODE = __STRICT_DIFFUSION_MODE__

# Optional: set a Tavily API key in the notebook config cell to use Tavily instead of
# the free DuckDuckGo search fallback.
TAVILY_API_KEY: Optional[str] = None

MAX_RESULTS = 8


# --------------------------------------------------------------------------------------
# 1. ORCHESTRATOR — entity extraction (Regex + dictionary/fuzzy matching, no LLM)
# --------------------------------------------------------------------------------------

class EntityExtractor:
    YEAR_PATTERN = re.compile(r"\b(19\d{2}|20\d{2})\b")

    # The knowledge graph stores genres in English (raw TMDB values, e.g. "Action"),
    # but prompts are frequently Persian. Fuzzy/substring matching Persian text
    # against English genre names can never work (different scripts), so genres are
    # matched via this explicit keyword dictionary FIRST, before falling back to the
    # generic Latin matcher for English-language prompts.
    PERSIAN_GENRE_MAP = {
        "اکشن": "Action",
        "ماجراجویی": "Adventure",
        "انیمیشن": "Animation",
        "کمدی": "Comedy",
        "جنایی": "Crime",
        "مستند": "Documentary",
        "درام": "Drama",
        "خانوادگی": "Family",
        "فانتزی": "Fantasy",
        "تخیلی": "Fantasy",
        "تاریخی": "History",
        "ترسناک": "Horror",
        "وحشت": "Horror",
        "موزیکال": "Music",
        "موسیقی": "Music",
        "معمایی": "Mystery",
        "رازآلود": "Mystery",
        "عاشقانه": "Romance",
        "رمانتیک": "Romance",
        "علمی تخیلی": "Science Fiction",
        "علمی‌تخیلی": "Science Fiction",
        "علمی-تخیلی": "Science Fiction",
        "تلویزیونی": "TV Movie",
        "هیجان انگیز": "Thriller",
        "هیجان‌انگیز": "Thriller",
        "تریلر": "Thriller",
        "جنگی": "War",
        "وسترن": "Western",
    }

    # Cue words that tell us whether a matched name should be treated as the ACTOR
    # or the DIRECTOR. This matters because some people (e.g. Tom Hanks) are BOTH an
    # Actor node and a Director node in the graph — without this disambiguation, a
    # prompt like "فیلمی با بازی Tom Hanks" would match him as actor AND director
    # simultaneously, and the Cypher query would then require a movie where he is
    # both at once (almost never true), silently returning zero results even though
    # he is clearly present in the graph.
    ACTOR_CUES = ["بازی", "بازیگر", "نقش", "با حضور", "starring", "actor", "played by", "with"]
    DIRECTOR_CUES = ["کارگردان", "کارگردانی", "ساخته", "directed by", "director", "made by"]

    def __init__(self, known_genres: List[str], known_actors: List[str],
                 known_directors: List[str]):
        self.known_genres = known_genres
        self.known_actors = known_actors
        self.known_directors = known_directors

        try:
            from rapidfuzz import process, fuzz
            self._process = process
            self._fuzz = fuzz
            self._fuzzy_available = True
        except ImportError:
            self._fuzzy_available = False
            print("[warn] rapidfuzz not installed — falling back to plain substring "
                  "matching for names/genres.")

        # spaCy is used as a lightweight, accurate NLP preprocessor: its NER model
        # tags PERSON spans directly (e.g. it recognizes "Tom Hanks" as one entity),
        # which is a more reliable candidate source for actor/director matching than
        # blindly sliding a word-window over the raw prompt. It only understands
        # English, so it's used as an extra signal alongside (not instead of) the
        # existing n-gram/fuzzy matcher, which still covers Persian-adjacent text.
        try:
            import spacy
            self._nlp = spacy.load("en_core_web_sm")
        except (ImportError, OSError):
            self._nlp = None
            print("[warn] spaCy/en_core_web_sm not available — name extraction will "
                  "rely only on the n-gram/fuzzy matcher (still functional).")

    def _spacy_preprocess(self, prompt: str) -> (List[str], List[str]):
        """Runs the full spaCy pipeline ONCE per prompt — tokenization, lemmatization,
        and stripping of punctuation/stopwords — BEFORE any matching against the
        graph's vocabulary takes place, as requested. Returns:
          - cleaned_tokens: lemmatized tokens with punctuation and English stopwords
            removed, used to build n-gram candidates for actor/director/genre
            matching (replaces the old naive prompt.split()).
          - person_entities: spaCy's PERSON-tagged spans (e.g. "Tom Hanks" as one
            unit) — a higher-precision candidate source than sliding n-grams.

        Note on Persian: en_core_web_sm has no Persian model, so Persian tokens
        simply pass through cleaned_tokens inertly (not flagged as stopwords, lemma
        unchanged) rather than being mishandled. For that reason, the Persian
        genre-keyword dictionary and the actor/director role-cue detection
        deliberately still scan the ORIGINAL raw prompt, not this English-oriented
        spaCy output — running Persian text through an English stopword/lemma filter
        would risk corrupting it for no benefit, since spaCy can't meaningfully
        lemmatize or POS-tag Persian anyway.
        """
        if self._nlp is None:
            return prompt.split(), []
        doc = self._nlp(prompt)
        cleaned_tokens = [
            tok.lemma_ for tok in doc
            if not tok.is_punct and not tok.is_space and not tok.is_stop
        ]
        person_entities = [ent.text for ent in doc.ents if ent.label_ == "PERSON"]
        return cleaned_tokens, person_entities

    def _exact_substring_match(self, prompt: str, choices: List[str]) -> Optional[str]:
        """Deterministic pass: does a known name literally appear in the prompt
        (case-insensitive)? Checked BEFORE fuzzy matching, zero false-positive risk.
        Length >= 4 avoids a short word matching almost anything."""
        prompt_lower = prompt.lower()
        best = None
        for choice in choices:
            if len(choice) >= 4 and choice.lower() in prompt_lower:
                if best is None or len(choice) > len(best):
                    best = choice
        return best

    def _best_fuzzy_match(self, text: str, choices: List[str], score_cutoff: int = 90
                           ) -> Optional[str]:
        if not choices:
            return None
        if self._fuzzy_available:
            # fuzz.ratio (whole-string Levenshtein similarity) is used instead of
            # fuzz.WRatio. WRatio blends in partial_ratio, which scores a short
            # chunk as a near-perfect match whenever it happens to be a substring
            # of a much longer candidate (e.g. "tom" scores ~100 against "Satomi
            # Ishihara" because "tom" is literally inside "Sa-TOM-i"). That caused
            # real false positives (e.g. "tom hanks" wrongly resolving to an
            # unrelated actor/director). fuzz.ratio does not do partial/substring
            # scoring, so short chunks no longer spuriously match unrelated names.
            match = self._process.extractOne(
                text, choices, scorer=self._fuzz.ratio, score_cutoff=score_cutoff
            )
            return match[0] if match else None
        text_lower = text.lower()
        for choice in choices:
            if choice.lower() in text_lower:
                return choice
        return None

    def _find_in_ngrams(self, cleaned_tokens: List[str], choices: List[str]) -> Optional[str]:
        """Matches against the spaCy-cleaned token stream (lemmatized, no
        punctuation/stopwords) instead of a naive prompt.split()."""
        # Pass 1: exact (case-insensitive) substring match over the cleaned tokens.
        exact = self._exact_substring_match(" ".join(cleaned_tokens), choices)
        if exact:
            return exact

        # Pass 2: fuzzy match, for typos / slightly different spelling only.
        for n in (4, 3, 2, 1):
            for i in range(len(cleaned_tokens) - n + 1):
                chunk = " ".join(cleaned_tokens[i:i + n])
                if len(chunk) < 4:
                    continue
                match = self._best_fuzzy_match(chunk, choices, score_cutoff=90)
                if match:
                    return match
        return None

    def _match_name(self, person_entities: List[str], cleaned_tokens: List[str],
                     choices: List[str]) -> Optional[str]:
        """Name matching for actor/director: tries spaCy's PERSON entities first
        (precise, whole-name spans), then falls back to the cleaned-token n-gram/
        fuzzy matcher (covers cases spaCy's NER misses, and non-English text)."""
        for candidate in person_entities:
            match = (self._exact_substring_match(candidate, choices)
                     or self._best_fuzzy_match(candidate, choices, score_cutoff=88))
            if match:
                return match
        return self._find_in_ngrams(cleaned_tokens, choices)

    def _match_genre(self, prompt: str, cleaned_tokens: List[str]) -> Optional[str]:
        normalized = prompt.replace("\u200c", " ")  # strip Persian ZWNJ (zero-width non-joiner)
        for fa_term, en_genre in self.PERSIAN_GENRE_MAP.items():
            if fa_term in normalized and en_genre in self.known_genres:
                return en_genre
        # English-language prompt fallback: match against the cleaned token stream.
        return self._find_in_ngrams(cleaned_tokens, self.known_genres)

    def _match_actor_and_director(self, prompt: str, cleaned_tokens: List[str],
                                   person_entities: List[str]) -> (Optional[str], Optional[str]):
        prompt_lower = prompt.lower()
        has_actor_cue = any(cue in prompt_lower for cue in self.ACTOR_CUES)
        has_director_cue = any(cue in prompt_lower for cue in self.DIRECTOR_CUES)

        if has_director_cue and not has_actor_cue:
            return None, self._match_name(person_entities, cleaned_tokens, self.known_directors)
        if has_actor_cue and not has_director_cue:
            return self._match_name(person_entities, cleaned_tokens, self.known_actors), None

        # No cue, or both cues present: check both lists independently.
        actor = self._match_name(person_entities, cleaned_tokens, self.known_actors)
        director = self._match_name(person_entities, cleaned_tokens, self.known_directors)
        if actor and director and actor == director and not has_director_cue:
            # Same person matched in both roles with no explicit director cue —
            # default to actor-only so the query isn't over-constrained.
            director = None
        return actor, director

    def extract(self, prompt: str) -> Dict[str, Any]:
        entities: Dict[str, Any] = {"year": None, "genre": None, "actor": None, "director": None}
        year_match = self.YEAR_PATTERN.search(prompt)
        if year_match:
            entities["year"] = int(year_match.group(1))

        # Single spaCy pass per prompt (tokenize + lemmatize + strip punctuation/
        # stopwords), BEFORE any matching against the graph vocabulary — its output
        # feeds every matcher below instead of each one re-splitting the raw string.
        cleaned_tokens, person_entities = self._spacy_preprocess(prompt)

        entities["genre"] = self._match_genre(prompt, cleaned_tokens)
        entities["actor"], entities["director"] = self._match_actor_and_director(
            prompt, cleaned_tokens, person_entities
        )
        return entities


# --------------------------------------------------------------------------------------
# 2 & 3. NEO4J LAYER + ROUTING
# --------------------------------------------------------------------------------------

class MovieGraphRAG:
    def __init__(self, uri: str, user: str, password: str):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))
        self._verify_connection()

    def _verify_connection(self):
        with self.driver.session() as session:
            session.run("RETURN 1").consume()

    def close(self):
        self.driver.close()

    def fetch_vocabulary(self) -> Dict[str, List[str]]:
        with self.driver.session() as session:
            genres = session.run("MATCH (g:Genre) RETURN g.name AS name").value("name")
            actors = session.run("MATCH (a:Actor) RETURN a.name AS name").value("name")
            directors = session.run("MATCH (d:Director) RETURN d.name AS name").value("name")
        return {
            "genres": [g for g in genres if g],
            "actors": [a for a in actors if a],
            "directors": [d for d in directors if d],
        }

    @staticmethod
    def build_cypher(entities: Dict[str, Any]) -> (str, Dict[str, Any]):
        match_clauses = ["MATCH (m:Movie)"]
        where_clauses = []
        params: Dict[str, Any] = {}

        if entities.get("genre"):
            match_clauses.append("MATCH (m)-[:HAS_GENRE]->(g:Genre)")
            where_clauses.append("toLower(g.name) = toLower($genre)")
            params["genre"] = entities["genre"]

        if entities.get("actor"):
            match_clauses.append("MATCH (a:Actor)-[:ActED_IN]->(m)")
            where_clauses.append("toLower(a.name) = toLower($actor)")
            params["actor"] = entities["actor"]

        if entities.get("director"):
            match_clauses.append("MATCH (d:Director)-[:DIRECTED]->(m)")
            where_clauses.append("toLower(d.name) = toLower($director)")
            params["director"] = entities["director"]

        if entities.get("year"):
            match_clauses.append("MATCH (m)-[:IN_YEAR]->(y:Year)")
            where_clauses.append("y.year = $year")
            params["year"] = entities["year"]

        query = "\n".join(match_clauses)
        if where_clauses:
            query += "\nWHERE " + " AND ".join(where_clauses)

        query += """
OPTIONAL MATCH (m)-[:HAS_GENRE]->(all_g:Genre)
OPTIONAL MATCH (all_a:Actor)-[:ActED_IN]->(m)
OPTIONAL MATCH (all_d:Director)-[:DIRECTED]->(m)
OPTIONAL MATCH (m)-[:IN_YEAR]->(all_y:Year)
OPTIONAL MATCH (m)-[wp:WITH_POINTS]->(:RATING)
RETURN DISTINCT
    m.title AS title,
    all_y.year AS year,
    wp.real_rating AS rating,
    collect(DISTINCT all_g.name) AS genres,
    collect(DISTINCT all_a.name) AS actors,
    all_d.name AS director
LIMIT $limit
"""
        params["limit"] = MAX_RESULTS
        return query, params

    def query_local_graph(self, entities: Dict[str, Any]) -> List[Dict[str, Any]]:
        query, params = self.build_cypher(entities)
        with self.driver.session() as session:
            result = session.run(query, params)
            return [record.data() for record in result]


class WebSearchFallback:
    def __init__(self, tavily_api_key: Optional[str] = None):
        self.tavily_api_key = tavily_api_key

    def search(self, query: str) -> List[Dict[str, str]]:
        if self.tavily_api_key:
            return self._search_tavily(query)
        return self._search_duckduckgo(query)

    def _search_tavily(self, query: str) -> List[Dict[str, str]]:
        try:
            from tavily import TavilyClient
        except ImportError:
            print("[warn] tavily-python not installed. Falling back to DuckDuckGo.")
            return self._search_duckduckgo(query)
        client = TavilyClient(api_key=self.tavily_api_key)
        response = client.search(query=query, max_results=5)
        return [{"title": r.get("title", ""), "snippet": r.get("content", "")}
                for r in response.get("results", [])]

    def _search_duckduckgo(self, query: str) -> List[Dict[str, str]]:
        try:
            from ddgs import DDGS
        except ImportError:
            try:
                from duckduckgo_search import DDGS
            except ImportError:
                print("[warn] no web search library installed (`pip install ddgs`).")
                return []
        results = []
        with DDGS() as ddgs:
            for r in ddgs.text(query, max_results=5):
                results.append({"title": r.get("title", ""), "snippet": r.get("body", "")})
        return results


# --------------------------------------------------------------------------------------
# 4. DIFFUSION LLM LAYER
# --------------------------------------------------------------------------------------

class DiffusionResponder:
    """
    Loads GSAI-ML/LLaDA-8B-Instruct, currently the best-documented free, open-weight
    diffusion LLM, and produces the final answer using it.

    This implementation was rewritten after checking the model's OFFICIAL repository
    (github.com/ML-GSAI/LLaDA) directly, because three earlier assumptions turned out
    to be wrong and were the real cause of repeated crashes:

      1. The official docs pin `transformers==4.38.2` — a much older version than
         what generic `pip install --upgrade transformers` installs. That huge
         version gap against the checkpoint's custom `trust_remote_code` is what was
         producing import errors deep inside `transformers`.
      2. The official loading example uses `AutoModel`, never `AutoModelForCausalLM`.
      3. LLaDA does NOT implement a standard `model.generate()`. It's a masked
         diffusion model — the official repo ships its own sampling loop
         (`generate.py`) that repeatedly calls the model's raw forward pass and
         iteratively "unmasks" tokens. There is no drop-in `.generate()` to call.

    `_diffusion_generate()` below reproduces that official sampling loop directly
    (mask-and-denoise with low-confidence remasking), so it calls the model in
    exactly the way its authors intended, instead of guessing at a HF-style API that
    this architecture never actually provides.

    Note: Dream-7B was considered as a second option but its own README explicitly
    states it "requires a GPU with at least 20GB memory" — more than the free Colab
    T4's 15GB — so it isn't a realistic free-tier choice and was dropped.
    """

    MODEL_NAME = "GSAI-ML/LLaDA-8B-Instruct"
    MASK_ID = 126336  # LLaDA's official [MASK] token id (documented in its model card)

    # Previously only one (Persian) template existed and was always used, even
    # if the user asked in English — and the model would sometimes repeat the
    # instructions/context above the prompt verbatim in the output instead of the
    # final answer. Now: (1) the Persian or English template is chosen based on
    # the language of the user's question, and the model is explicitly asked to
    # answer in that same language and only with the final answer text; (2) a
    # specific marker ("پاسخ نهایی:" / "Final answer:") is appended at the end of the
    # prompt, so if part of the prompt leaks into the model's output (a known
    # behavior of diffusion models that are weak at instruction-following), it can
    # be reliably located and stripped out in _clean_answer.
    PERSIAN_CHAR_RE = re.compile(r"[\u0600-\u06FF]")

    PROMPT_TEMPLATE_FA = (
        "زمینهٔ (context) زیر را فقط برای پیدا کردن اطلاعات لازم بخوان؛ آن را و این "
        "دستورالعمل را عیناً در پاسخ کپی نکن.\n\n"
        "زمینه:\n{context}\n\n"
        "سوال کاربر: {user_prompt}\n\n"
        "با توجه به زمینهٔ بالا، یک پاسخ نهایی کوتاه، دوستانه و پیشنهادی فقط به "
        "زبان فارسی بنویس (نه دستورالعمل بالا، نه متن زمینه را تکرار نکن):\n"
        "پاسخ نهایی:"
    )

    PROMPT_TEMPLATE_EN = (
        "Read the context below only to find the information you need; do not copy "
        "it or these instructions verbatim into your reply.\n\n"
        "Context:\n{context}\n\n"
        "User question: {user_prompt}\n\n"
        "Based on the context above, write a short, friendly, suggestive final "
        "answer in English only (do not repeat the instructions above or the "
        "context text):\n"
        "Final answer:"
    )

    ANSWER_MARKERS = ["پاسخ نهایی:", "Final answer:"]

    def __init__(self, enabled: bool, strict: bool = False):
        self.enabled = enabled
        self.strict = strict
        self.tokenizer = None
        self.model = None
        self.loaded_model_name: Optional[str] = None
        self._load_attempted = False

    def _lazy_load(self):
        if self._load_attempted or not self.enabled:
            return
        self._load_attempted = True

        import torch
        if not torch.cuda.is_available():
            msg = ("GPU is not available; the 8-billion-parameter diffusion model "
                   "won't be loaded on CPU (it needs an impractical amount of "
                   "memory). To enable it, use Runtime > Change runtime type > GPU "
                   "in Colab.")
            if self.strict:
                raise RuntimeError(msg)
            print(f"[warn] {msg} Falling back to the template-based response.")
            return

        # Proactive RAM check: loading an 8B checkpoint briefly needs several GB of
        # *system* RAM while shards are read/converted (more so with
        # low_cpu_mem_usage=False, used below as a fallback path — see note there).
        # On free-tier Colab (~12GB RAM) insufficient headroom can silently OOM-kill
        # the kernel — bypassing try/except entirely (the process is killed, not a
        # catchable Python exception). Skipping the attempt avoids crashing outright.
        try:
            import psutil
            available_gb = psutil.virtual_memory().available / (1024 ** 3)
            if available_gb < 9:
                msg = (f"Only ~{available_gb:.1f} GB of RAM is free — risk of a "
                       f"kernel crash if loading the model is attempted. Run "
                       f"Runtime > Restart session and run this cell earlier "
                       f"(with more free RAM).")
                if self.strict:
                    raise RuntimeError(msg)
                print(f"[warn] {msg} Falling back to the template-based response.")
                return
        except ImportError:
            pass

        from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig

        print(f"[info] Loading diffusion model '{self.MODEL_NAME}' (4-bit, this "
              f"may take a few minutes, please keep the Colab tab open)...")
        # bnb_4bit_compute_dtype=float16 (not bfloat16): the free-tier T4 is a
        # Turing-generation GPU without native bf16 tensor-core support, so fp16 is
        # the hardware-appropriate compute dtype here.
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
        )
        self.tokenizer = AutoTokenizer.from_pretrained(self.MODEL_NAME, trust_remote_code=True)

        # Two-attempt loading strategy. Attempt 1 uses low_cpu_mem_usage=True (the
        # default transformers applies automatically for quantized models) — this is
        # RAM-lean but, with LLaDA's custom trust_remote_code class specifically, can
        # trigger `ValueError: .to is not supported for 4-bit or 8-bit bitsandbytes
        # models` from inside from_pretrained's own meta-tensor dispatch step (a bug
        # in how this particular custom model class interacts with that codepath —
        # not something fixable via from_pretrained kwargs alone). If that exact
        # error occurs, attempt 2 retries with low_cpu_mem_usage=False, which avoids
        # meta-tensor loading entirely (the model materializes normally on CPU first,
        # then bitsandbytes quantizes it onto the GPU) — more system RAM, but sidesteps
        # the buggy codepath.
        last_error = None
        for attempt, low_cpu_mem_usage in enumerate([True, False], start=1):
            try:
                self.model = AutoModel.from_pretrained(
                    self.MODEL_NAME,
                    trust_remote_code=True,
                    quantization_config=bnb_config,
                    low_cpu_mem_usage=low_cpu_mem_usage,
                )
                self.model.eval()
                self.loaded_model_name = self.MODEL_NAME
                print(f"[info] ✅ Diffusion model '{self.MODEL_NAME}' loaded successfully "
                      f"(attempt {attempt}, low_cpu_mem_usage={low_cpu_mem_usage}).")
                return
            except ValueError as e:
                if "not supported for" in str(e) and attempt == 1:
                    print(f"[warn] Attempt {attempt} (low_cpu_mem_usage=True) hit a "
                          f"known compatibility error — retrying with "
                          f"low_cpu_mem_usage=False...")
                    last_error = e
                    continue
                last_error = e
                break
            except Exception as e:
                last_error = e
                break

        # Both attempts failed.
        self.model = None
        self.tokenizer = None
        if self.strict:
            raise RuntimeError(f"Loading the diffusion model failed after both attempts: {last_error}") from last_error
        print("[warn] Loading the diffusion model failed — falling back to the template-based response.")
        traceback.print_exc(limit=1)

    @staticmethod
    def _add_gumbel_noise(logits, temperature):
        import torch
        if temperature == 0:
            return logits
        logits = logits.to(torch.float64)
        noise = torch.rand_like(logits, dtype=torch.float64)
        gumbel_noise = (-torch.log(noise)) ** temperature
        return logits.exp() / gumbel_noise

    @staticmethod
    def _get_num_transfer_tokens(mask_index, steps):
        import torch
        mask_num = mask_index.sum(dim=1, keepdim=True)
        base = mask_num // steps
        remainder = mask_num % steps
        num_transfer_tokens = (
            torch.zeros(mask_num.size(0), steps, device=mask_index.device, dtype=torch.int64) + base
        )
        for i in range(mask_num.size(0)):
            num_transfer_tokens[i, :remainder[i]] += 1
        return num_transfer_tokens

    def _diffusion_generate(self, prompt_ids, gen_length=96, block_length=32,
                             steps=48, temperature=0.7, remasking="low_confidence"):
        """Reproduces LLaDA's official masked-diffusion sampling loop (generate.py in
        github.com/ML-GSAI/LLaDA): starts from an all-[MASK] canvas after the prompt
        and iteratively fills in the most confident tokens block-by-block, rather
        than generating left-to-right like an autoregressive model."""
        import torch
        import torch.nn.functional as F

        device = self.model.device
        x = torch.full((1, prompt_ids.shape[1] + gen_length), self.MASK_ID,
                        dtype=torch.long, device=device)
        x[:, :prompt_ids.shape[1]] = prompt_ids.to(device)
        prompt_len = prompt_ids.shape[1]

        assert gen_length % block_length == 0
        num_blocks = gen_length // block_length
        assert steps % num_blocks == 0
        steps_per_block = steps // num_blocks

        # Memory-bug fix: the previous version ran the entire sampling loop without
        # torch.no_grad(), meaning PyTorch kept the autograd graph and activations
        # for every one of these repeated forward passes (up to 48, each over the
        # full sequence) — designed for a backward pass that is never actually used
        # here. On a T4 with ~15GB VRAM, this alone could cause an OOM. inference_mode
        # removes this overhead entirely without changing the sampling logic.
        with torch.inference_mode():
            for b in range(num_blocks):
                block_start = prompt_len + b * block_length
                block_end = prompt_len + (b + 1) * block_length
                block_mask_index = (x[:, block_start:block_end] == self.MASK_ID)
                num_transfer_tokens = self._get_num_transfer_tokens(block_mask_index, steps_per_block)

                for i in range(steps_per_block):
                    mask_index = (x == self.MASK_ID)
                    logits = self.model(x).logits
                    logits_with_noise = self._add_gumbel_noise(logits, temperature=temperature)
                    x0 = torch.argmax(logits_with_noise, dim=-1)

                    if remasking == "low_confidence":
                        p = F.softmax(logits.to(torch.float64), dim=-1)
                        x0_p = torch.squeeze(torch.gather(p, dim=-1, index=torch.unsqueeze(x0, -1)), -1)
                    else:
                        x0_p = torch.rand(x0.shape, device=device)

                    x0_p[:, block_end:] = -float("inf")
                    x0 = torch.where(mask_index, x0, x)
                    confidence = torch.where(mask_index, x0_p, torch.tensor(-float("inf"), device=device))

                    transfer_index = torch.zeros_like(x0, dtype=torch.bool)
                    k = int(num_transfer_tokens[0, i].item())
                    if k > 0:
                        _, select_index = torch.topk(confidence[0], k=k)
                        transfer_index[0, select_index] = True
                    x[transfer_index] = x0[transfer_index]

                # Memory-bug fix: clear the CUDA allocator cache between blocks so
                # freed memory chunks (each step's temporary logits) are actually
                # returned to the system, so VRAM doesn't gradually fill up over a
                # long REPL session.
                if device.type == "cuda":
                    torch.cuda.empty_cache()

        return x[:, prompt_len:]

    def _run_model(self, final_prompt: str) -> str:
        messages = [{"role": "user", "content": final_prompt}]
        prompt_ids = self.tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt"
        )
        output_ids = self._diffusion_generate(prompt_ids)
        text = self.tokenizer.decode(output_ids[0], skip_special_tokens=True)
        # Memory-bug fix: after every full answer, clear the VRAM cache once more —
        # important for a REPL loop that may process dozens of prompts in a row in
        # a single Colab session.
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return text.strip()

    def respond(self, user_prompt: str, context: str) -> (str, Optional[str]):
        """Returns (answer_text, model_name_used_or_None). model_name_used is None
        when the template fallback was used instead of the diffusion model."""
        is_persian = bool(self.PERSIAN_CHAR_RE.search(user_prompt))
        template = self.PROMPT_TEMPLATE_FA if is_persian else self.PROMPT_TEMPLATE_EN
        final_prompt = template.format(context=context, user_prompt=user_prompt)
        self._lazy_load()
        if self.model is not None and self.tokenizer is not None:
            try:
                raw = self._run_model(final_prompt)
                answer = self._clean_answer(raw, context)
                return answer, self.loaded_model_name
            except Exception:
                if self.strict:
                    raise
                print("[warn] diffusion model inference failed — using template fallback.")
                traceback.print_exc(limit=1)
        elif self.strict:
            raise RuntimeError("The diffusion model is not loaded and STRICT_DIFFUSION_MODE "
                                "is enabled — per the configuration, the template-based "
                                "fallback response is not allowed.")
        return self._fallback_response(context, is_persian), None

    @classmethod
    def _clean_answer(cls, raw: str, context: str) -> str:
        """The diffusion model sometimes repeats part of the instructions or the
        context text itself in its output instead of just the final answer. This
        function cleans up that leak: first it discards everything before the last
        occurrence of the "پاسخ نهایی:"/"Final answer:" marker (since the most
        genuine answer is usually right after the last time the model produced that
        marker), then, if the context text still appears verbatim in the output, it
        removes that too."""
        text = raw.strip()

        for marker in cls.ANSWER_MARKERS:
            idx = text.rfind(marker)
            if idx != -1:
                text = text[idx + len(marker):].strip()

        if context and context.strip() and context.strip() in text:
            text = text.replace(context.strip(), "").strip()

        text = re.sub(r"\n{3,}", "\n\n", text).strip()
        return text or raw.strip()

    @staticmethod
    def _fallback_response(context: str, is_persian: bool = True) -> str:
        if is_persian:
            return (
                "بر اساس اطلاعاتی که پیدا کردم:\n"
                f"{context}\n\n"
                "(پاسخ بالا با الگوی جایگزین ساخته شد چون مدل دیفیوژن در دسترس/قابل‌لود نبود.)"
            )
        return (
            "Based on what I found:\n"
            f"{context}\n\n"
            "(This answer was generated with the fallback template because the diffusion model wasn't available/loadable.)"
        )


# --------------------------------------------------------------------------------------
# Context formatting helpers
# --------------------------------------------------------------------------------------

def format_local_results(rows: List[Dict[str, Any]]) -> str:
    lines = []
    for row in rows:
        title = row.get("title") or "Unknown"
        year = row.get("year")
        rating = row.get("rating")
        genres = ", ".join(g for g in (row.get("genres") or []) if g)
        actors = ", ".join(a for a in (row.get("actors") or []) if a)
        director = row.get("director") or "Unknown"

        parts = [f"- «{title}»"]
        if year:
            parts.append(f"({year})")
        if rating:
            parts.append(f"| Rating: {rating}")
        if director and director != "none":
            parts.append(f"| Director: {director}")
        if genres:
            parts.append(f"| Genre: {genres}")
        if actors:
            parts.append(f"| Actors: {actors}")
        lines.append(" ".join(parts))
    return "\n".join(lines)


def format_web_results(results: List[Dict[str, str]]) -> str:
    return "\n".join(f"- {r['title']}: {r['snippet']}" for r in results)


# --------------------------------------------------------------------------------------
# 5. HTTP API LAYER
# --------------------------------------------------------------------------------------
# This section replaces the previous interactive loop (input()/print() in the
# terminal). The logic for processing each prompt is exactly the same as before
# (extract entities -> query the local graph -> if empty, web search -> generate
# an answer with the diffusion model); only the input/output layer changed from
# stdin/stdout to an HTTP/JSON API so the chat page (which opens in your own
# browser, separate from Colab) can talk to it.
# --------------------------------------------------------------------------------------

from flask import Flask, request, jsonify
from flask_cors import CORS

app = Flask(__name__)
# CORS is left open because the HTML chat page sends requests from a different
# origin (a local file or anywhere else), not from Colab itself.
CORS(app)

_graph: Optional["MovieGraphRAG"] = None
_extractor: Optional["EntityExtractor"] = None
_web_search: Optional["WebSearchFallback"] = None
_responder: Optional["DiffusionResponder"] = None


def init_pipeline() -> None:
    """Exactly the same initialization that used to happen at the start of main()."""
    global _graph, _extractor, _web_search, _responder

    try:
        _graph = MovieGraphRAG(NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD)
    except Exception:
        print("[error] Could not connect to Neo4j. Please check the "
              "URI/USERNAME/PASSWORD and make sure the Neo4j service is running.")
        traceback.print_exc()
        raise

    try:
        vocab = _graph.fetch_vocabulary()
        if not vocab["genres"] and not vocab["actors"] and not vocab["directors"]:
            print("[warn] The local graph appears to be empty (no Genre/Actor/Director "
                  "found). Make sure you ran the graph-building cells before this "
                  "one — otherwise all answers will come from web search.")
    except Exception:
        print("[warn] Failed to read the vocabulary from the graph; entity extraction will be more limited.")
        vocab = {"genres": [], "actors": [], "directors": []}

    _extractor = EntityExtractor(vocab["genres"], vocab["actors"], vocab["directors"])
    _web_search = WebSearchFallback(tavily_api_key=TAVILY_API_KEY)
    _responder = DiffusionResponder(DIFFUSION_ENABLED, strict=STRICT_DIFFUSION_MODE)

    # Previous issue: the diffusion model was only loaded lazily, on the first real
    # user request, so that very first question had to wait several minutes for the
    # 8-billion-parameter weights to download/load — which usually exceeded the
    # public tunnel's (Cloudflare) timeout too and showed the user an error, even
    # though the model itself had actually loaded, and the second question onward
    # got a fast response. To fix this, the model is now loaded right here, before
    # the server announces it's ready (and before any request arrives).
    if DIFFUSION_ENABLED:
        print("⏳ Loading the diffusion model (this may take a few minutes)...")
        _responder._lazy_load()
        if _responder.model is not None:
            print(f"✅ Diffusion model '{_responder.loaded_model_name}' loaded.")
        elif STRICT_DIFFUSION_MODE:
            raise RuntimeError("The diffusion model failed to load and STRICT_DIFFUSION_MODE is enabled.")
        else:
            print("[warn] The diffusion model failed to load — answers will be built with the fallback template.")

    print("✅ The GraphRAG pipeline is ready — the server is ready to receive prompts from the chat page.")


def process_query(user_prompt: str) -> Dict[str, Any]:
    """The exact processing body of the previous REPL loop, with no logic changes —
    only returns a structured dictionary instead of using print."""
    entities = _extractor.extract(user_prompt)

    rows = _graph.query_local_graph(entities)

    if rows:
        source = "local_graph"
        context = format_local_results(rows)
    else:
        web_results = _web_search.search(user_prompt)
        source = "web_search" if web_results else "none"
        context = format_web_results(web_results) if web_results else \
            "Unfortunately, no information was found, either in the local database or on the web."

    answer, used_model = _responder.respond(user_prompt, context)

    return {
        "entities": entities,
        "source": source,
        "result_count": len(rows),
        "answer": answer,
        "used_model": used_model,
    }


@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok"})


@app.route("/chat", methods=["POST"])
def chat():
    data = request.get_json(force=True, silent=True) or {}
    user_prompt = (data.get("message") or "").strip()

    if not user_prompt:
        return jsonify({"error": "The message is empty."}), 400

    try:
        result = process_query(user_prompt)
        return jsonify(result)
    except Exception:
        traceback.print_exc()
        return jsonify({"error": "An error occurred while processing your request. Check the Colab log."}), 500


def main():
    init_pipeline()
    try:
        # threaded=True: so the /health request isn't blocked while the diffusion
        # model is processing a request.
        app.run(host="0.0.0.0", port=5000, threaded=True)
    finally:
        if _graph is not None:
            _graph.close()
        print("[info] Neo4j connection closed. Goodbye!")


if __name__ == "__main__":
    main()


In [ ]:
# Automatically substitute the Neo4j password and diffusion model settings inside the script
with open("graphrag_movie_cli.py", "r", encoding="utf-8") as f:
    content = f.read()

content = content.replace('__NEO4J_PASSWORD__', NEO4J_PASSWORD_COLAB)
content = content.replace('__DIFFUSION_ENABLED__', str(DIFFUSION_ENABLED_COLAB))
content = content.replace('__STRICT_DIFFUSION_MODE__', str(STRICT_DIFFUSION_MODE_COLAB))

with open("graphrag_movie_cli.py", "w", encoding="utf-8") as f:
    f.write(content)

import os
size_kb = os.path.getsize("graphrag_movie_cli.py") / 1024
print(f"✅ Script graphrag_movie_cli.py written and configured ({size_kb:.1f} KB).")
print(f"   NEO4J_PASSWORD set to: {NEO4J_PASSWORD_COLAB}")
print(f"   DIFFUSION_ENABLED set to: {DIFFUSION_ENABLED_COLAB}")
print(f"   STRICT_DIFFUSION_MODE set to: {STRICT_DIFFUSION_MODE_COLAB}")


In [ ]:
# Final GPU check right before running — make sure the session hasn't
# lost its GPU since the start of the notebook (e.g. due to a long idle period).
import torch
if not torch.cuda.is_available():
    print("🛑 GPU is no longer connected! Go back to Section 0, check "
          "Runtime > Change runtime type, and continue from there. Otherwise the "
          "diffusion model will not load and you'll only get the template-based response.")
else:
    print(f"✅ GPU is connected ({torch.cuda.get_device_name(0)}) — ready to run.")


In [ ]:
# Install the HTTP server prerequisites and the public tunnel tool (no account/signup needed)
!pip install -q flask flask-cors
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
print("✅ Flask and cloudflared installed.")


In [ ]:
import threading, subprocess, re, time, urllib.request
import graphrag_movie_cli

def _start_flask():
    graphrag_movie_cli.init_pipeline()
    graphrag_movie_cli.app.run(host="0.0.0.0", port=5000, threaded=True)

flask_thread = threading.Thread(target=_start_flask, daemon=True)
flask_thread.start()

print("⏳ The Flask server is starting up (including loading the diffusion model on the GPU; this may take a few minutes)...")
for _ in range(300):
    try:
        urllib.request.urlopen("http://127.0.0.1:5000/health", timeout=1)
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("The Flask server did not come up within the wait period — check the output of the previous cells.")
print("✅ The Flask server is running on port 5000.")

tunnel_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:5000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)

public_url_holder = {"url": None}

def _drain_tunnel_output():
    for line in tunnel_proc.stdout:
        if public_url_holder["url"] is None:
            m = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", line)
            if m:
                public_url_holder["url"] = m.group(0)

drain_thread = threading.Thread(target=_drain_tunnel_output, daemon=True)
drain_thread.start()

print("⏳ Creating the public tunnel (Cloudflare Quick Tunnel)...")
for _ in range(60):
    if public_url_holder["url"]:
        break
    time.sleep(1)

if not public_url_holder["url"]:
    raise RuntimeError("The public tunnel address was not found — run this cell again.")

print("=" * 70)
print("🌐 Your server's public address is ready:")
public_url = public_url_holder["url"]
print(f"   {public_url}")
print()
print("Enter this address on the chat page (filmyar-chat.html, which you open in your own browser)")
print("in the \"Connect to server\" button.")
print("=" * 70)
print("⚠️ Do not stop (Stop/Interrupt) this cell — the server and tunnel stay alive only while it's running.")

flask_thread.join()  # deliberately keeps the cell alive so the server stays up
